In [1]:
import os
import dill
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold


from sklearn.metrics import (auc,roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, brier_score_loss,
                             roc_curve, precision_recall_curve)
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

import shap

In [ ]:
all_IV = pd.read_csv('/2025data/IV_mdro_basic_37180.csv')
print(all_IV.shape)
all_IV.MDR_LABEL.value_counts()

(37180, 52)


0    33707
1     3473
Name: MDR_LABEL, dtype: int64

In [3]:
missing_cols = all_IV.columns[all_IV.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [ ]:
old_IV = pd.read_csv('/MIMIC_IV/Median_B/Final/All_40815_df_minmax_24.csv')
old_IV = old_IV[old_IV.first_icu_stay == 1]
print(old_IV.shape)

(37180, 250)


In [6]:
old_IV = old_IV[['ICUSTAY_ID','Age', 'gender','admission_type','first_hosp_stay', 'first_careunit','admin_counts','A40', 'D64', 'D69', 'E11', 'E66', 'E78', 'E8497', 'E87', 'E8798', 'E89', 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47', 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09', 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41', 'CHART_220045', 'CHART_220179', 'CHART_220180', 'CHART_220181', 'CHART_220210', 'CHART_220277', 'CHART_223834', 'CHART_226253', 'CHART_224639', 'CHART_220224', 'CHART_220228', 'CHART_220235', 'CHART_223830', 'CHART_225667', 'CHART_225668', 'CHART_220545', 'CHART_220546', 'CHART_220602', 'CHART_220615', 'CHART_220645', 'CHART_225624', 'CHART_225625', 'CHART_227073', 'CHART_227443', 'CHART_227457', 'CHART_227465', 'CHART_227466', 'CHART_227467', 'CHART_227442', 'CHART_220587', 'CHART_220621', 'CHART_220644', 'CHART_225612', 'CHART_225690', 'CHART_220227', 'CHART_220274', 'CHART_227429', 'CHART_220603', 'CHART_220632', 'CHART_229357', 'CHART_229358', 'CHART_229359', 'CHART_229360', 'CHART_229361', 'CHART_220050', 'CHART_220051', 'CHART_220052', 'CHART_220339', 'CHART_224690', 'CHART_224697', 'CHART_224700', 'CHART_220074', 'CHART_227456', 'CHART_226534', 'CHART_226536', 'CHART_226537', 'CHART_226540', 'CHART_227464', 'CHART_225651', 'CHART_224144', 'CHART_223762', 'CHART_225638', 'CHART_229355', 'CHART_Cate_225092', 'CHART_Cate_225113', 'CHART_Cate_225103', 'CHART_Cate_225091', 'CHART_Cate_225112', 'CHART_Cate_225118', 'CHART_Cate_225126', 'CHART_Cate_225094', 'CHART_Cate_225124', 'CHART_Cate_225108', 'CHART_Str_220739', 'CHART_Str_223900', 'CHART_Str_223901', 'CHART_Str_225101', 'CHART_Str_228699', 'LAB_50802', 'LAB_51249', 'LAB_51277', 'LAB_51279', 'LAB_52172', 'LAB_51516', 'LAB_50811', 'LAB_51493', 'LAB_51300', 'LAB_Str_51463', 'PROC_229351', 'PROC_225752', 'PROC_224264', 'PROC_225792', 'PROC_224270', 'PROC_225794', 'PROC_225802', 'PROC_225441', 'PROC_224268', 'PROC_224272', 'PROC_225805', 'PROC_224273', 'PROC_Cate_225402', 'PROC_Cate_225401', 'PROC_Cate_221214', 'PROC_Cate_221217', 'PROC_Cate_227194', 'PROC_Cate_224385', 'PROC_Cate_221216', 'PROC_Cate_227712', 'PROC_Cate_225454', 'PROC_Cate_225966', 'PROC_Cate_225444', 'PROC_Cate_225440', 'PROC_Cate_225439', 'PROC_Cate_223253', 'PROC_Cate_225427', 'PROC_Cate_225400', 'PROC_Cate_225451', 'PROC_Cate_225817', 'PROC_Cate_221223', 'PROC_Cate_225816', 'PROC_Cate_225814', 'PROC_Cate_225445', 'PROC_Cate_225466', 'PROC_Cate_225464', 'PROC_Cate_225399', 'PROC_Cate_225430', 'PROC_Cate_225434', 'PROC_Cate_225448', 'PROC_Cate_228715', 'PROC_Cate_225433', 'PROC_Cate_225479', 'PROC_Cate_225437', 'PROC_Cate_225475', 'PROC_Cate_227713', 'PROC_Cate_225467', 'PROC_Cate_225449', 'PROC_Cate_225472', 'PROC_Cate_225450', 'PROC_Cate_225967', 'PROC_Cate_225442', 'MED_221749', 'MED_221744', 'MED_222168', 'MED_225168', 'MED_223258', 'MED_221906', 'MED_225154', 'MED_221468', 'MED_222042', 'MED_221794', 'MED_222315', 'MED_222056', 'MED_225152', 'MED_221662', 'MED_225170', 'MED_221289', 'MED_221986', 'MED_221555', 'MED_221653', 'MED_221319', 'MED_229630', 'MED_221282', 'MED_229617']]

In [7]:
all_IV = pd.merge(all_IV,old_IV,on='ICUSTAY_ID',how='left')

In [8]:
missing_cols = all_IV.columns[all_IV.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [ ]:
ventilation = pd.read_csv('/ventilation.csv')
ventilation.columns = ['ICUSTAY_ID', 'starttime', 'endtime', 'ventilation_status']
ventilation = pd.merge(ventilation,all_IV[['ICUSTAY_ID','ADMITTIME','INTIME', 'OUTTIME']],
                       on='ICUSTAY_ID',how='left')
ventilation = ventilation.dropna(subset=['ADMITTIME'])
datetime_cols = ['ADMITTIME', 'INTIME','OUTTIME', 'starttime']
ventilation[datetime_cols] = ventilation[datetime_cols].apply(pd.to_datetime)

ventilation_icu_window_criteria = ventilation['starttime'].between(
    ventilation['ADMITTIME'],
    ventilation['INTIME'] + pd.DateOffset(1))
ventilation_icu = ventilation[ventilation_icu_window_criteria]
InvasiveVent = ventilation_icu[ventilation_icu.ventilation_status == 'InvasiveVent']
InvasiveVent_ids = list(InvasiveVent.ICUSTAY_ID.unique())
InvasiveVent_ids = [int(i) for i in InvasiveVent_ids]
all_IV['InvasiveVent'] = all_IV['ICUSTAY_ID'].isin(InvasiveVent_ids).astype(int)

In [10]:
id_mapping = {
    "CHART_225612": "Alkaline Phosphate",
    "CHART_224700": "Total PEEP Level",
    "CHART_227457": "Platelet Count",
    "CHART_225625": "Calcium non-ionized",
    "LAB_51249": "MCHC",
    "CHART_220181": "Non-invasive Blood Pressure mean",
    "CHART_226537": "Glucose (whole blood)",
    "LAB_51516": "WBC",
    "CHART_225651": "Direct Bilirubin",
    "CHART_227073": "Anion gap",
    "MED_221906": "Norepinephrine",
    "CHART_225624": "BUN",
    "CHART_225690": "Total Bilirubin",
    "CHART_220545": "Hematocrit (serum)",
    "CHART_220228": "Hemoglobin",
    "CHART_220277": "O2 saturation pulse",
    "CHART_220210": "Respiratory Rate",
    "CHART_225668": "Lactic Acid",
    "CHART_Str_225101": "Use of assistive devices",
    "CHART_227466": "PTT",
    "LAB_51279": "Red Blood Cells",
    "CHART_220235": "Arterial CO2 Pressure",
    "LAB_51277": "RDW",
    "CHART_220179": "Non-invasive Blood Pressure systolic",
    "CHART_220045": "Heart Rate",
    "CHART_220615": "Creatinine (serum)",
    "CHART_227465": "Prothrombin time",
    "CHART_227443": "HCO3 (serum)",
    "PROC_224270": "Dialysis Catheter",
    'PROC_225802':'Dialysis - CRRT',
    'CHART_223762':'Temperature',
    'PROC_225792':'Invasive Ventilation',
    "PROC_225752": "Arterial Line",
    'CHART_Cate_225113':'Experiencing pain',
     'CHART_Cate_225092':'Self ADL',
    #'T78': 'Adverse effects',
    'N18': 'CKD',#Chronic kidney disease (CKD)
    'J18': 'Pneumonia',
    #'I95': 'Hypotension',
    'E11': 'Type 2 Diabetes',#Type 2 diabetes mellitus
    'J98': 'Respiratory Disorders',
    'D64': 'Anemias',
    #'F10': 'Alcohol',
    'A40': 'Streptococcal sepsis',
    'J44': 'COPD ',#chronic obstructive pulmonary disease
    'N39': 'Urinary Disorders',#Other disorders of urinary system
    'gender':'Gender',
    'ab_id_count':'Antibiotics use',
    'ATE':'AET',
    'admission_type':'Admission type',
    'first_hosp_stay':'First hosp stay', 
    'first_careunit':'First careunit',
    'admin_counts':'Hosp counts', 
    'MDR_before':'MDROs before',
    'BAL':'BAL Culture', 'CSF':'CSF Culture', 
    'ABSCESS':'Abscess Culture', 'PERITONEAL FLUID':'Peritoneal Fluid Culture', 
    'SPUTUM':'Sputum Culture',  'URINE':'Urine Culture', 
    'MRSA SCREEN':'MRSA Screen','BLOOD CULTURE':'Blood Culture',
    'SWAB':'Swab',
    
    "PROC_225794": "Non-invasive Ventilation",
    "MED_221468": "Diltiazem",
    "LAB_50802": "Base Excess",
    "MED_221289": "Epinephrine",
    "CHART_224144": "Blood Flow (ml/min)",
    "PROC_224273": "Presep Catheter",
    "CHART_220644": "ALT",
    "MED_221986": "Milrinone",
    "PROC_Cate_225816": "Wound Culture",
    "CHART_224639": "Daily Weight",
    "MED_223258": "Insulin - Regular",
    "CHART_226253": "SpO2 Desat Limit",
    "CHART_229357": "Absolute Count - Neuts",
    "PROC_Cate_225444": "Pan Culture",
    "CHART_Str_223901": "GCS - Motor Response",
    "MED_222168": "Propofol",
    "CHART_229360": "Absolute Count - Eos",
    #"PROC_Cate_225451": "Sputum Culture",
    #"PROC_Cate_225445": "Paracentesis",
    "MED_225170": "Platelets",
    "MED_225168": "Packed Red Blood Cells",
    "CHART_220602": "Chloride (serum)",
    "MED_221319": "Alteplase (TPA)",
    "PROC_225805": "Peritoneal Dialysis",
    "MED_229617": "Epinephrine.",
    #"PROC_229351": "Foley Catheter",
    "PROC_224268": "Trauma line",
    "MED_229630": "Phenylephrine (50/250)",
    "PROC_Cate_225479": "Thoracentesis",
    "CHART_220587": "AST",
    "PROC_Cate_225400": "Bronchoscopy",
    "PROC_224272": "IABP line",
    "CHART_220180": "Non Invasive Blood Pressure diastolic",
    "CHART_225667": "Ionized Calcium",
    "CHART_227467": "INR",
    "CHART_226534": "Sodium (whole blood)",
    "MED_222042": "Nicardipine",
    "CHART_229359": "Absolute Count - Monos",
    "CHART_227442": "Potassium (serum)",
    "PROC_Cate_225399": "Lumbar Puncture",
    "CHART_223830": "PH (Arterial)",
    #"PROC_Cate_225401": "Blood Cultured",
    "CHART_220052": "Arterial Blood Pressure mean",
    "CHART_227456": "Albumin",
    "MED_221662": "Dopamine",
    "CHART_220274": "PH (Venous)",
    "CHART_223834": "O2 Flow",
    "CHART_220051": "Arterial Blood Pressure diastolic",
    "MED_221282": "Adenosine",
    "LAB_52172": "RDW-SD",
    "MED_221794": "Furosemide (Lasix)",
    "CHART_227429": "Troponin-T",
    "CHART_220645": "Sodium (serum)",
    "CHART_220339": "PEEP set",
    "LAB_51493": "RBC",
    "CHART_220050": "Arterial Blood Pressure systolic",
    "LAB_Str_51463": "Bacteria",
    "CHART_220621": "Glucose (serum)",
    "MED_225154": "Morphine Sulfate",
    "CHART_224690": "Respiratory Rate (Total)",
    "PROC_225441": "Hemodialysis",
    "MED_225152": "Heparin Sodium",
    "MED_222056": "Nitroglycerin",
    "CHART_229355": "Absolute Neutrophil Count",
    "CHART_229361": "Absolute Count - Basos",
    "LAB_51300": "WBC Count",
    "PROC_Cate_225454": "Urine Culture",
    "CHART_220227": "Arterial O2 Saturation",
    "CHART_226540": "Hematocrit (whole blood - calc)",
    "CHART_227464": "Potassium (whole blood)",
    "PROC_Cate_225817": "BAL Fluid Culture",
    "CHART_220603": "Cholesterol",
    "CHART_220074": "Central Venous Pressure",
    "CHART_224697": "Mean Airway Pressure",
    "MED_221653": "Dobutamine",
    "MED_222315": "Vasopressin",
    "CHART_220632": "LDH",
    "MED_221749": "Phenylephrine",
    "MED_221555": "Cisatracurium",
    #'CHART_Cate_225118':'Difficulty swallowing',
    'CHART_226536':'Chloride (whole blood)',
#     'CHART_Cate_225094':'History of slips / falls',
    'CHART_220224':'PO2 (Arterial)',
    'CHART_Cate_225124':'Unintentional weight loss >10 lbs.',
    'CHART_225638':'Differential-Bands',
    'MED_221744':'Fentanyl',
    'PROC_Cate_224385':'Intubation',
    'PROC_224264':'PICC Line'
#     'CHART_Cate_225108':'Tobacco use'
}

In [ ]:
models = [
         ]

In [12]:
C_P_M_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/d_items.csv.gz')
D_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_icd_diagnoses.csv.gz')
L_id = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/d_labitems.csv.gz')
diag_col = ['A40', 'D64', 'D69', 'E11', 'E66', 'E78', 'E8497', 'E87', 'E8798', 'E89', 'F05', 'F10', 'F32', 'F41', 'G47', 'I10', 'I12', 'I21', 'I25', 'I27', 'I34', 'I47', 'I50', 'I95', 'J18', 'J44', 'J69', 'J98', 'K22', 'N17', 'N18', 'N39', 'R00', 'R09', 'R68', 'R78', 'T78', 'T81', 'Z51', 'Z91', 'Z95', 'A41']

In [ ]:
all_IV.rename(columns=id_mapping)[['ICUSTAY_ID', 'FIRST_CAREUNIT','MDR_BEFORE','MRSA Screen','ATE_FILTERED','Gender','RACE','AGE','ADMISSION_TYPE','Use of assistive devices','InvasiveVent']].to_csv('/2025data/HeterogeneousSubset.csv',index=False)

In [153]:
IV_run = all_IV.copy()
labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']

#'icu_counts','first_icu_stay','ab_id_filtered_count','ATE_filtered', 
f_b = ['Age', 'gender','admission_type','first_hosp_stay', 'first_careunit','admin_counts',
      'AB_ID_COUNT','ATE_FILTERED', 'MDR_BEFORE']

f_d = ['CHART_Cate_225092','CHART_Cate_225113','N18', 'J18',  'E11', 'J98', 'D64','A40', 'J44','N39']

f_cul = ['BAL', 'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM',  'URINE', 'MRSA SCREEN','BLOOD CULTURE', 'SWAB']
#IV_run[f_cul] = IV_run[f_cul].apply(lambda x: (x != 0).astype(int))

ft_s = ['CHART_220545', 'LAB_51249', 'LAB_51279', 'CHART_225624', 
        'CHART_220228',  'CHART_223762', 'CHART_225612', 'CHART_220615', 'LAB_51277', 
        'CHART_220179', 'CHART_Str_225101', 'LAB_51516', 'CHART_220235', 'CHART_225651', 
        'CHART_226537', 'CHART_227073', 'CHART_220277', 'CHART_220045', 'CHART_225690', 'MED_221906', 
        'CHART_225625', 'CHART_227443', 'CHART_227465', 'CHART_220210',  'CHART_227466', 
        'CHART_224700', 'CHART_227457', 'CHART_225668',
        'PROC_225802', 'PROC_224270', 'PROC_225792','PROC_225752']
len(ft_s)

32

In [154]:
features = ft_s + f_b + f_d + f_cul
print(len(features))

60


In [155]:
all_df = IV_run[features+labels + ['RACE','AGE','ADMISSION_TYPE','FIRST_CAREUNIT','InvasiveVent']].rename(columns=id_mapping)

In [157]:
features = list(set(all_df.columns.tolist())-set(labels+['RACE','AGE','ADMISSION_TYPE','FIRST_CAREUNIT','InvasiveVent']))
len(features)

60

In [159]:
cultures = ['BAL Culture', 'CSF Culture', 'Abscess Culture', 'Peritoneal Fluid Culture', 'Sputum Culture', 'Urine Culture', 'MRSA Screen', 'Blood Culture', 'Swab']

In [29]:
import baseM

In [ ]:
#labels = ['ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL']
this_sub = IV_gender1.copy()
this_l = 'MDR_LABEL'
print(len(features))
print(Counter(this_sub[this_l]),this_sub.shape,f'{round((len(this_sub[this_sub[this_l]==1])/this_sub.shape[0])*100,1)}%')
Train, Test = train_test_split(this_sub, test_size=0.3, random_state=42)
Lable_fi = features + [this_l]
print('Train:',Counter(Train[this_l]))
print('Test:',Counter(Test[this_l]))

all_AUROC, all_AUPRC, all_scores,all_importances = baseM.BaseModel.bootstrap(Train[Lable_fi], Test[Lable_fi], this_l, models, n=20, confidence=0.95)
all_scores.index = ['LR', 'RF', 'XGB']
all_scores.style.highlight_max(color='lightgreen', axis=0)